# Milestone 5 Lab, notebook 6 of 7: A swarm of file workers

## Where we are

| Notebook | What you do |
|---|---|
| 1. Files arrive, by hand | Load the files the naive way, and watch three things go wrong |
| 2. A loader that is safe to repeat | Fingerprint each file, check its columns, and choose a rule for every file |
| 3. AI reads a changed file | Ask a model what a renamed column means, and score its suggestions |
| 4. A workflow that can wait | Build the loader as a graph that pauses for your approval |
| 5. Crash and carry on | Restart the kernel on purpose, and find the file still waiting |
| **6. A swarm of file workers** (you are here) | Load fifty files side by side, and score the run for quality, cost, and time |
| 7. The same question, from loaded files | Approve the waiting files, run the pipeline, and ask the leader's question |

Model requests in this notebook: **about 18**. A full run of this lab costs about 3 US cents.

## What this notebook is for

So far you loaded files one at a time. A real morning brings fifty at once, and the ones that need a model take seconds each.

In this notebook you build a **dispatcher**: a small graph that hands every file to its own worker and runs several workers side by side. People call many agents working at once a **swarm**.

The dispatcher has no AI in it. So you build and test it first with a pretend worker that costs nothing. Then you give it the real file workflow, and score each run for quality, cost, and time.

> This notebook needs `OPENROUTER_API_KEY`, and the Langfuse settings from milestone 4 for the last part.

In [ ]:
import operator
import time
from datetime import datetime
from typing import Annotated, TypedDict

import pandas as pd
from lab_helpers import LAB, LANDING, count_outcomes, reset_lab, show_doc, show_timeline

reset_lab()

## Step 1: The ideas

In [ ]:
show_doc("HLD.md", "8. Many files at once")

## Step 2: The dispatcher's state

The dispatcher keeps two things: the files to load, and the results that come back.

`results` has something new: `Annotated[list, operator.add]`. Several workers finish at the same moment, and each returns a list with its own result. `operator.add` tells LangGraph how to combine them: add the lists together. Without it, one worker's result would replace another's.

In [ ]:
class Batch(TypedDict):
    """What the dispatcher keeps for one batch of files."""

    files: list[str]
    results: Annotated[list, operator.add]

## Step 3: Fanning out

`fan_out` is a conditional edge. In notebook 4 a conditional edge returned the name of one node. This one returns a **list of `Send`**, one per file. A `Send` names the node to run and the input to give it. So `work_file` runs once for every file.

In [ ]:
from langgraph.types import Send


def fan_out(state: Batch):
    """Send each file to its own worker."""
    return [Send("work_file", file_name) for file_name in state["files"]]

## Step 4: A stand-in for the worker

Before spending anything, we give the dispatcher a worker that only pretends. It sleeps for a fifth of a second and records when it started and finished.

In [ ]:
def stand_in(file_name):
    """Pretend to load a file for a fifth of a second."""
    started = datetime.now()
    time.sleep(0.2)
    return {"results": [{"file_name": file_name, "started": started, "finished": datetime.now()}]}

## Step 5: Build the dispatcher

Two edges: from the start, fan out to the workers. From each worker, go to the end.

In [ ]:
from langgraph.graph import END, START, StateGraph

builder = StateGraph(Batch)
builder.add_node("work_file", stand_in)
builder.add_conditional_edges(START, fan_out, ["work_file"])
builder.add_edge("work_file", END)
dispatcher = builder.compile()

print(dispatcher.get_graph().draw_mermaid())

There is no model in this graph. LangGraph is a way to run any workflow with parallel steps, retries, and saved state, whether or not it has AI in it.

## Step 6: A cap on how many run at once

`max_concurrency` in the config caps how many nodes run at the same time. Try one at a time first.

In [ ]:
files = sorted(file.name for file in LANDING.glob("*.csv"))


def seconds_taken(results):
    """From the first worker starting to the last one finishing."""
    table = pd.DataFrame(results)
    return (table["finished"].max() - table["started"].min()).total_seconds()


result = dispatcher.invoke({"files": files}, {"max_concurrency": 1})
print(f"1 at once:  {len(result['results'])} files in {seconds_taken(result['results']):.1f} seconds")

In [ ]:
result = dispatcher.invoke({"files": files}, {"max_concurrency": 5})
print(f"5 at once:  {len(result['results'])} files in {seconds_taken(result['results']):.1f} seconds")
five_at_once = result["results"]

In [ ]:
result = dispatcher.invoke({"files": files}, {"max_concurrency": 50})
print(f"50 at once: {len(result['results'])} files in {seconds_taken(result['results']):.1f} seconds")

Fifty files at a fifth of a second each is ten seconds one at a time, two seconds with five at once, and about a fifth of a second with all fifty at once.

Here is the run with five at once. Each bar is a file, from when its worker started to when it finished.

In [ ]:
show_timeline(five_at_once, "The stand-in, 5 at once")

Look down any vertical line: it never crosses more than five bars. As soon as one worker finishes, the next file starts.

## Step 7: A retry

When many workers call a model at once, some calls fail: the provider answers "429 Too Many Requests", or times out. Without a retry, one failed file stops the whole batch.

A `RetryPolicy` on a node runs it again after a short wait. This stand-in fails the first time it meets one file.

In [ ]:
tries = {}


def flaky_stand_in(file_name):
    """The stand-in, failing the first time it works one of the files."""
    tries[file_name] = tries.get(file_name, 0) + 1
    if file_name == "shipments_2026-10-26.csv" and tries[file_name] == 1:
        raise RuntimeError("429 Too Many Requests")
    return stand_in(file_name)

In [ ]:
from langgraph.types import RetryPolicy

builder = StateGraph(Batch)
builder.add_node("work_file", flaky_stand_in, retry_policy=RetryPolicy(max_attempts=3, retry_on=Exception))
builder.add_conditional_edges(START, fan_out, ["work_file"])
builder.add_edge("work_file", END)
flaky_dispatcher = builder.compile()

result = flaky_dispatcher.invoke({"files": files}, {"max_concurrency": 50})

print(len(result["results"]), "files done")
print("Tries for the flaky file:", tries["shipments_2026-10-26.csv"])

The flaky file was tried twice, and all fifty files were done.

A retry is only safe because of notebook 5: a worker that runs twice loads nothing twice.

## Step 8: The real workers

Now the same dispatcher with the real file workflow as its worker. Production's `load_batch` is this dispatcher. Its `work_file` runs the file workflow from notebook 4 for one file, **on that file's own thread**.

That last part matters. Each file keeps its own saved state. So one file can pause for a person while the other 49 finish.

In [ ]:
from retail_profit_stack.agent import make_model
from retail_profit_stack.ingest.dispatch import load_batch

model = make_model(timeout=30)

`score_batch` runs one batch from a clean start and scores it. It clears the lab's folders and its approved mappings first, so every run has to ask the model about the same six files.

In [ ]:
from retail_profit_stack.ingest.mapping import score_suggestions


def score_batch(at_once, callbacks=None):
    """Load every file from a clean start, `at_once` at a time, and score the run."""
    reset_lab()
    results = load_batch(
        places=LAB, model=model, max_concurrency=at_once, callbacks=callbacks, run_name=f"Swarm of {at_once}"
    )

    suggested = {}
    for item in results:
        if item["outcome"] == "waiting":
            columns = item["suggestion"]["columns"]
            suggested[item["file_name"]] = {
                guess["file_column"]: guess["expected_column"]
                for guess in columns
                if guess["expected_column"] != "none"
            }
    scored = score_suggestions(suggested)

    scores = {
        "files": len(results),
        "waiting": count_outcomes(results).get("waiting", 0),
        "right": int((scored["result"] == "right").sum()),
        "model_cost": round(sum(item["model_cost"] for item in results), 4),
        "seconds": round(seconds_taken(results), 1),
        "slowest_file": max(item["seconds"] for item in results),
    }
    return results, scores

## Step 9: One at a time

This is how notebooks 3 to 5 worked. It takes about twenty seconds.

In [ ]:
one_results, one_scores = score_batch(at_once=1)

print(count_outcomes(one_results))
one_scores

In [ ]:
show_timeline(one_results, "The real workers, 1 at once")

Most bars are too thin to see: a file that loads by rule takes a few hundredths of a second. The six long bars are the files that asked the model. They are the whole cost of the run, in time and in money.

## Step 10: Five at once, with a trace

This run is sent to Langfuse. We pass Langfuse's handler in the config, as in milestone 4. LangGraph hands the config on to every worker, so every model call of the batch goes into **one trace**.

In [ ]:
from lab_helpers import connect_langfuse
from langfuse.langchain import CallbackHandler

langfuse = connect_langfuse()
langfuse_handler = CallbackHandler()

In [ ]:
five_results, five_scores = score_batch(at_once=5, callbacks=[langfuse_handler])
five_trace = langfuse_handler.last_trace_id

show_timeline(five_results, "The real workers, 5 at once")

## Step 11: All at once

In [ ]:
all_results, all_scores = score_batch(at_once=50, callbacks=[langfuse_handler])
all_trace = langfuse_handler.last_trace_id

show_timeline(all_results, "The real workers, 50 at once")

## Step 12: The scoreboard

In [ ]:
scoreboard = pd.DataFrame([one_scores, five_scores, all_scores], index=["1 at once", "5 at once", "50 at once"])
scoreboard

When the lab was built, the three runs took about 16, 5, and 4 seconds, and each cost about a fifth of a US cent.

What the scoreboard shows:

- **The swarm saves time.** One at a time, the six model calls queue behind each other. Side by side, they overlap.
- **The slowest file sets the clock.** With everything at once, the run takes about as long as its slowest file. If one of your runs took minutes, a model call stalled and was tried again. It happened once while this lab was being built: one file took two and a half minutes, and so did the run. Run that cell again.
- **The cost stays about the same.** Each run makes the same six model calls, at any number at once.
- **Quality comes from the worker, not the dispatcher.** `right` is the number of suggestions that match the truth. It can move a little from run to run. The number at once does not change it.
- **The outcomes are the same.** Every run ends with 42 loaded, 1 skipped, 1 rejected, and 6 waiting. Running side by side changed nothing about what was loaded.

The scores go onto each run's trace in Langfuse with `create_score`.

In [ ]:
def send_scores(trace_id, scores):
    """Add a run's scores to its trace in Langfuse."""
    for name, value in scores.items():
        langfuse.create_score(trace_id=trace_id, name=name, value=float(value))


send_scores(five_trace, five_scores)
send_scores(all_trace, all_scores)
langfuse.flush()

print("Swarm of 5: ", langfuse.get_trace_url(trace_id=five_trace))
print("Swarm of 50:", langfuse.get_trace_url(trace_id=all_trace))

Open the two links. In each trace, switch to the **timeline** view. You see the six model calls side by side, where milestone 4's traces showed one call after another. The scores appear on the trace.

## Step 13: Same records, in any order

One worry remains. With fifty workers finishing in any order, do we get the same records? Compare the tables from the last run with a run one at a time.

In [ ]:
from retail_profit_stack.transform.pipeline import load_sources

side_by_side = {name: table.num_rows for name, table in load_sources(LAB.loaded).items()}

reset_lab()
load_batch(places=LAB, max_concurrency=1)
one_by_one_tables = load_sources(LAB.loaded)
one_by_one = {name: table.num_rows for name, table in one_by_one_tables.items()}

print("Side by side:", side_by_side)
print("One by one:  ", one_by_one)

The same. Each file becomes its own piece, and a table is its pieces read **in file-name order**. So the order the workers finished in cannot matter. The production tests compare every row, not only the counts.

The six renamed files are missing from both, which is why `order_items` is short. They are waiting for you in the next notebook.

## What we now know

- A **dispatcher** hands each file to its own worker. `Send` starts a worker; `max_concurrency` caps how many run at once.
- It has no AI in it. We built and tested it with a stand-in before spending anything.
- A **retry** runs a failed worker again. That is only safe because loading is safe to repeat.
- Side by side saves time. It does not change cost, quality, or the records.
- Each file runs on its own thread, so one file can wait for a person while the rest finish.

**Next:** notebook 7. Six files are still waiting. You approve them, run the whole pipeline on what was loaded, and ask the leader's question.